<a href="https://colab.research.google.com/github/mlbyhoang/marketing-attribution/blob/main/notebooks/01_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 – Data Preparation

This notebook prepares customer journey data for the marketing attribution project.

The raw data contains advertising interactions and CM360 activities.
The goal of this notebook is to clean and structure the data for:

1. Conversion prediction
2. Multi-touch attribution

In [ ]:
# Load ibraries
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Connect to Google Drive
from google.colab import drive, userdata
drive.mount('/content/drive')

In [ ]:
# Load data
DATA_PATH = userdata.get("MTA_DATA_PATH")

df = pd.read_csv(DATA_PATH, delimiter=",")
df.head()

In [ ]:
# Show data types and columns
df.info()

In [ ]:
# Show rows and columns
df.shape

In [ ]:
# Rename columns
df.rename(columns={"floodlight_variable_dimension21": "ga_cid"}, inplace=True)
df.head()

In [ ]:
# Drop irrelevant columns
df2 = df.drop(columns=["conversion_id", "impression_count", "click_count", "placement", "activity_count"])
df2.head()

In [ ]:
# Convert time features to datetime
df2["interaction_time"] = pd.to_datetime(df2["interaction_time"], format="%Y-%m-%d %H:%M:%S")
df2["activity_time"] = pd.to_datetime(df2["activity_time"], format="%Y-%m-%d %H:%M:%S")

In [ ]:
# Map anonymized campaign identifiers to marketing channels
campaign_mapping = {
    "Campaign_001": "paid_search",
    "Campaign_002": "organic_search",
    "Campaign_003": "affiliate",
    "Campaign_004": "display",
    "Campaign_005": "affiliate",
    "Campaign_006": "native",
    "Campaign_007": "display",
    "Campaign_008": "twitch"
}

df2["channel"] = (
    df2["campaign"]
    .map(campaign_mapping)
    .fillna("unknown")
)

In [ ]:
# Show columns, rows, and data types
df2.info()

In [ ]:
# Check for missing values
df2.isnull().sum()

In [ ]:
# Filter rows where 'ga_cid' is null
df_nan_cid = df2[df2["ga_cid"].isna()]
df_nan_cid.head()

In [ ]:
# Check shape of null 'ga_cid' rows
df_nan_cid.shape

In [ ]:
# Check shape of df2
df2.shape

In [ ]:
# Drop rows with missing 'ga_cid' values
df3 = df2[df2["ga_cid"].notna()].reset_index(drop=True)
df3.shape

In [ ]:
# Check for duplicates
df3.duplicated().sum()

In [ ]:
# Overview of duplicate rows
print(df3.duplicated(keep=False).sum(), "duplizierte Zeilen")
print(df3.duplicated().sum(), "zu entfernende Zeilen")

In [ ]:
# Drop duplicates
df4 = df3.drop_duplicates(keep="first")
df4.shape

In [ ]:
# Define target variable (sale / order)
order_users       = set(df4[df4["activity"] == "Order_Server"]["ga_cid"])
server_sale_users = set(df4[df4["activity"] == "Server_Sale"]["ga_cid"])

overlap = order_users & server_sale_users

print(f"Order_Server Users:     {len(order_users)}")
print(f"Server_Sale Users:      {len(server_sale_users)}")
print(f"User in beiden:         {len(overlap)}")
print(f"Anteil an Server_Sale in Overlap:  {len(overlap) / len(server_sale_users):.2%}")
print(f"Anteil Order_Server in Overlap: {len(overlap) / len(order_users):.2%}")

order_times = df4[df4["activity"] == "Order_Server"].groupby("ga_cid")["activity_time"].min()
sale_times  = df4[df4["activity"] == "Server_Sale"].groupby("ga_cid")["activity_time"].min()

comparison = pd.DataFrame({
    "order_time": order_times,
    "sale_time":  sale_times
}).dropna()

comparison["order_before_sale"] = comparison["order_time"] < comparison["sale_time"]
print(f"\nOrder vor Sale:  {comparison['order_before_sale'].mean():.2%}")
print(f"Sale vor Order:  {(~comparison['order_before_sale']).mean():.2%}")

### Target variable logic:

*   Order AND Server_Sale -> Server_Sale
*   Only Order_Server    -> Order_Server
*   Only Server_Sale     -> Server_Sale
*   Neither Order/Sale   -> 0

In [ ]:
# Verify sale/order
conversion_users = server_sale_users | order_users

print(f"Server_Sale Users: {len(server_sale_users)}")
print(f"Order_Server Users: {len(order_users)}")
print(f"Conversion Users: {len(conversion_users)}")

In [ ]:
# Check order and timestamp
conversion_tp = (
    df4[df4['activity'].isin(['Order_Server', 'Server_Sale'])]
    .groupby('ga_cid')['interaction_time']
    .apply(set)
    .to_dict()
)

print(f"User mit Conversion-Timestamps: {len(conversion_tp)}")

In [ ]:
# Display first 5 rows of df4
df4.head()

In [ ]:
# Check maximum interaction time
print(df4["interaction_time"].min(), df4["interaction_time"].max())

In [ ]:
# Feature engineering: time features
df4["interaction_hour"] = df4["interaction_time"].dt.hour
df4["interaction_minute"] = df4["interaction_time"].dt.minute
df4["interaction_weekday"] = df4["interaction_time"].dt.weekday

df4["interaction_hour_sin"] = np.sin(2 * np.pi * df4["interaction_hour"] / 24)
df4["interaction_hour_cos"] = np.cos(2 * np.pi * df4["interaction_hour"] / 24)
df4["interaction_minute_sin"] = np.sin(2 * np.pi * df4["interaction_minute"] / 60)
df4["interaction_minute_cos"] = np.cos(2 * np.pi * df4["interaction_minute"] / 60)
df4["interaction_weekday_sin"] = np.sin(2 * np.pi * df4["interaction_weekday"] / 7)
df4["interaction_weekday_cos"] = np.cos(2 * np.pi * df4["interaction_weekday"] / 7)

In [ ]:
# Verify time features
df4.head()

In [ ]:
# Time since first touchpoint per user (in minutes)
df4["tp_time_delta_min"] = df4.groupby("ga_cid")["interaction_time"].transform(
    lambda x: (x - x.min()).dt.total_seconds() / 60
)

In [ ]:
# Create binary flag (1.0 for click, 0.0 for other interactions)
df4["is_click"] = (
    df4["interaction_type"] == "Click"
).astype(float)

print(f"Click-Rate: {df4['is_click'].mean():.1%}")

In [ ]:
# Sort df4 by "ga_cid" and "interaction_time"
df4 = df4.sort_values(["ga_cid", "interaction_time"])
df4.head()

In [ ]:
# Display columns
df4.columns

In [ ]:
# Check unique touchpoints
print(f"Zeilen df4: {len(df4)}")
print(f"Unique Touchpoints: {df4.groupby(['ga_cid', 'interaction_time', 'channel']).ngroups}")

In [ ]:
# Deduplicate at touchpoint level
df4_dedup = df4.drop_duplicates(
    subset=['ga_cid', 'interaction_time', 'channel'],
    keep='first'
).sort_values(['ga_cid', 'interaction_time'])

print(f"Vorher:  {len(df4)} Zeilen")
print(f"Nachher: {len(df4_dedup)} Zeilen")
print(f"Unique Touchpoints: {df4_dedup.groupby(['ga_cid', 'interaction_time', 'channel']).ngroups}")

In [ ]:
# Define features
time_feature_cols = [
    # Interaction time features
    "interaction_hour_sin",
    "interaction_hour_cos",
    "interaction_weekday_sin",
    "interaction_weekday_cos",

    # Journey features
    "tp_time_delta_min",
    "interaction_number",

    # Interaction type
    "is_click"
]

# Derived feature groups
sincos_cols   = [
    k for k in time_feature_cols
    if '_sin' in k or '_cos' in k
    ]

journey_cols  = [
    "tp_time_delta_min",
    "interaction_number"
    ]

binary_cols = [
    "is_click"
    ]

print(f"Anzahl time_feature_cols: {len(time_feature_cols)}")
print(f"Sin/Cos Features:         {len(sincos_cols)}")
print(f"Journey Features:         {len(journey_cols)}")
print(f"Binary Features:          {len(binary_cols)}")

In [ ]:
# Sequence building
sequences = []

for user, group in df4_dedup.groupby("ga_cid"):

    group = group.sort_values("interaction_time")

    seq = []
    seq_times = []
    seq_start_time = None

    user_conv_tps = conversion_tp.get(user, set())

    for _, row in group.iterrows():

        # Start time of the current journey
        if seq_start_time is None:
            seq_start_time = row["interaction_time"]

        # Time since first touchpoint of the current journey
        tp_time_delta_min = (
            row["interaction_time"] - seq_start_time
        ).total_seconds() / 60

        # Position within the current observed journey
        interaction_number = len(seq) + 1

        # Channel sequence
        seq.append(row["channel"])

        # Touchpoint-level features
        tp_features = {
            col: row[col]
            for col in time_feature_cols
            if col not in ["tp_time_delta_min", "interaction_number"]
        }

        # Add journey-specific features
        tp_features["tp_time_delta_min"] = tp_time_delta_min
        tp_features["interaction_number"] = interaction_number

        seq_times.append(tp_features)

        # Conversion reached -> close journey
        if row["interaction_time"] in user_conv_tps:

            sequences.append({
                "ga_cid": user,
                "sequence": seq.copy(),
                "seq_times": seq_times.copy(),
                "touchpoint_sum": len(seq),
                "order": 1
            })

            # Start a new journey after the conversion
            seq = []
            seq_times = []
            seq_start_time = None

    # Remaining journey without conversion
    if seq:
        sequences.append({
            "ga_cid": user,
            "sequence": seq.copy(),
            "seq_times": seq_times.copy(),
            "touchpoint_sum": len(seq),
            "order": 0
        })

df5 = pd.DataFrame(sequences)

In [ ]:
# Safety checks
all_channels = [tp for seq in df5["sequence"] for tp in seq]
unique_channels = sorted(set(all_channels))

# Check click rate in seq_times
click_rate_seq = df5['seq_times'].apply( lambda times: np.mean([t['is_click'] for t in times]) ).mean()
print("Shape df5:", df5.shape)
print(f"Conversion Rate: {df5['order'].mean():.2%}")
print(f"Click-Rate (Ø): {click_rate_seq:.1%}")
print(f"Längen stimmen überein: {df5.apply(lambda r: len(r['sequence']) == len(r['seq_times']), axis=1).all()}")
print(f"Max Sequenzlänge: {df5['touchpoint_sum'].max()}")
print(f"Mittlere Sequenzlänge: {df5['touchpoint_sum'].mean():.2f}")
print(f"Unique Channels: {len(unique_channels)}")
print(f"Channels: {unique_channels}")
print(df5['touchpoint_sum'].describe())

In [ ]:
# Save processed dataset
PROCESSED_DATA_PATH = userdata.get("MTA_PROCESSED_PATH")

df5.to_parquet(
    PROCESSED_DATA_PATH,
    index=False
)

print("Processed dataset saved successfully.")